# 검증 데이터 분석: 안전모 미착용

두 모델의 val 결과를 확인하고 탐지 기준값과 오류 사례를 살펴봅니다. test는 사용하지 않습니다.


In [ ]:
!pip -q install ultralytics==8.4.208
import json, zipfile, yaml, torch
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
drive.mount("/content/drive")


In [ ]:
ROOT = Path("/content/drive/MyDrive/ppe-project")
ZIP = Path("/content/drive/MyDrive/construction-ppe-5class.zip")
DATA = Path("/content/ppe_project_data/construction-ppe-5class")
assert ZIP.is_file(), f"ZIP을 찾지 못했습니다: {ZIP}"
if not (DATA / "split_manifest.csv").is_file():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        assert z.testzip() is None
        z.extractall(DATA.parent)
config = yaml.safe_load((DATA / "data.yaml").read_text())
config["path"] = str(DATA)
YAML = DATA / "data_colab.yaml"
YAML.write_text(yaml.safe_dump(config, sort_keys=False))
assert len(list((DATA / "images/val").glob("*.jpg"))) == 100
WEIGHTS = {"yolo11n": ROOT / "runs/baseline_yolo11n_seed42/weights/best.pt", "yolo26n": ROOT / "runs/proposed_yolo26n_seed42/weights/best.pt"}
for name, weight in WEIGHTS.items():
    assert weight.is_file(), f"가중치가 없습니다: {weight}"
    print(name, weight)


## 기준값별 미탐 분석 (val만 사용)
IoU 0.5 이상을 같은 객체로 매칭합니다. 빨간 박스는 정답 중 미탐, 초록 박스는 탐지된 정답입니다. 이 방식의 수치는 기존 요약 Recall과 산출 방식이 다를 수 있습니다.


In [ ]:
import csv
from PIL import Image, ImageDraw

def iou(a, b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1])
    x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union else 0

OUT = ROOT / "val_analysis" / "no_helmet_threshold_study"
OUT.mkdir(parents=True, exist_ok=True)
thresholds=[0.05,0.10,0.20,0.25,0.30,0.40,0.50]
all_rows=[]
for name, weight in WEIGHTS.items():
    model=YOLO(str(weight))
    totals={t:[0,0,0] for t in thresholds}
    missed=[]
    for image in sorted((DATA / "images/val").glob("*.jpg")):
        with Image.open(image) as im: width,height=im.size
        label=DATA / "labels/val" / (image.stem+".txt")
        gt=[]
        for line in label.read_text().splitlines():
            a=line.split()
            if int(a[0])==2:
                x,y,w,h=map(float,a[1:]);gt.append(((x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height))
        pred=model.predict(str(image), conf=0.001, imgsz=640, device=0, verbose=False)[0]
        detections=sorted([(float(b.conf[0]),tuple(map(float,b.xyxy[0].tolist()))) for b in pred.boxes if int(b.cls[0])==2],reverse=True)
        for t in thresholds:
            used=set();tp=fp=0
            for conf,box in detections:
                if conf<t: continue
                matches=[(iou(box,g),j) for j,g in enumerate(gt) if j not in used]
                best=max(matches,default=(0,-1))
                if best[0]>=0.5: used.add(best[1]);tp+=1
                else: fp+=1
            fn=len(gt)-len(used)
            totals[t][0]+=tp;totals[t][1]+=fp;totals[t][2]+=fn
            if t==0.25 and fn:
                missed.append((image.name,fn,len(gt)))
                target=OUT / (name+"_missed_at_025");target.mkdir(exist_ok=True)
                with Image.open(image) as original:
                    vis=original.convert("RGB");draw=ImageDraw.Draw(vis)
                    for j,box in enumerate(gt):
                        draw.rectangle(box,outline="lime" if j in used else "red",width=6)
                    vis.thumbnail((1280,1280));vis.save(target/image.name,quality=90)
    for t,(tp,fp,fn) in totals.items():
        row={"model":name,"threshold":t,"TP":tp,"FP":fp,"FN":fn,"precision":round(tp/(tp+fp),4) if tp+fp else 0,"recall":round(tp/(tp+fn),4) if tp+fn else 0}
        all_rows.append(row);print(row)
    with (OUT/(name+"_missed_images.csv")).open("w",newline="") as f:
        w=csv.writer(f);w.writerow(["image","missed_no_helmet_boxes","total_no_helmet_boxes"]);w.writerows(missed)
with (OUT/"threshold_metrics.csv").open("w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=["model","threshold","TP","FP","FN","precision","recall"]);w.writeheader();w.writerows(all_rows)
print("저장 폴더:",OUT)
